# DEMO DIVRS — Khử thiên lệch (ML-10M)

So sánh **DIVRS** với baseline **Most-Popular** (luôn gợi ý đồ phổ biến nhất — đại diện cho thiên lệch đám đông).
**KHÔNG train, KHÔNG cần GPU** — chỉ nạp weight DIVRS đã có trên Drive.
Gồm: (A) bảng số, (B) đường cong độ-phổ-biến, (C) **web demo Gradio**.

## 1. Setup: code + data + Drive

In [ ]:
!pip install -q faiss-cpu gradio matplotlib
import os, glob, re
REPO='/content/DIVRS'
if os.path.exists(REPO):
    !cd {REPO} && git fetch -q origin && git reset --hard -q origin/main
else:
    !git clone -q https://github.com/HatDuaa/DIVRS-reproduce.git {REPO}
%cd {REPO}
os.makedirs('data', exist_ok=True)
if not os.path.exists('data/ml10m/output/train_coo_record.npz'):
    !cd data && wget -q https://raw.githubusercontent.com/tsinghua-fib-lab/DICE/main/data/ml10m.zip && unzip -oq ml10m.zip
from google.colab import drive; drive.mount('/content/drive')
OUT_DIR='/content/drive/MyDrive/Colab Notebooks/DIVRS_output'

## 2. Nạp embedding DIVRS (checkpoint best trên Drive) + dữ liệu

In [ ]:
import numpy as np, scipy.sparse as sp, torch
D='data/ml10m/output/'
train=sp.load_npz(D+'train_coo_record.npz').tocsr()
test =sp.load_npz(D+'test_coo_record.npz').tocsr()
pop  =np.load(D+'popularity.npy').astype(float)
n_user,n_item=train.shape
pop_pct=pop.argsort().argsort()/(len(pop)-1)   # 0=it pho bien, 1=pho bien nhat

def best_ckpt(run):
    cks=glob.glob(run+'ckpt/epoch_*.pth')
    best_ep,best_rc=None,-1
    for lf in glob.glob(run+'log/*'):
        try: txt=open(lf,encoding='utf-8',errors='ignore').read()   # bo qua symlink .INFO hong tren Drive
        except Exception: continue
        for m in re.finditer(r"VALIDATION epoch: (\d+).*?'recall': np\.float64\(([\d.]+)\)", txt):
            ep,rc=int(m.group(1)),float(m.group(2))
            if rc>best_rc: best_rc,best_ep=rc,ep
    if best_ep is not None and os.path.exists(run+f'ckpt/epoch_{best_ep}.pth'):
        return run+f'ckpt/epoch_{best_ep}.pth', best_ep, round(best_rc,4)
    mx=max(cks,key=lambda x:int(re.findall(r'epoch_(\d+)',x)[0]))
    return mx, int(re.findall(r'epoch_(\d+)',mx)[0]), None

# chon run DIVRS co NHIEU checkpoint nhat (tranh run bi interrupt)
div_runs=[r for r in glob.glob(OUT_DIR+'/*/') if 'divrs' in r.lower() and glob.glob(r+'ckpt/epoch_*.pth')]
assert div_runs, 'Khong thay run DIVRS nao co checkpoint tren Drive! Train xong it nhat 1 epoch da.'
div_run=max(div_runs, key=lambda r: len(glob.glob(r+'ckpt/epoch_*.pth')))
ckpt,ep,rc=best_ckpt(div_run)
sd=torch.load(ckpt,map_location='cpu')
Ud=sd['users_iv'].float().numpy(); Id=sd['items_iv'].float().numpy()
print('DIVRS run:',div_run)
print('Dung checkpoint: epoch',ep,'| val recall:',rc)
print('embedding:',Ud.shape,Id.shape,'| users',n_user,'items',n_item)

def score_divrs(u): return Ud[u] @ Id.T
def score_pop(u):   return pop                      # ai cung nhu nhau: theo do pho bien
MODELS={'Most-Popular (baseline)':score_pop, 'DIVRS (debiased)':score_divrs}

## A. Bảng so sánh: Recall@20 & Độ phổ biến TB của gợi ý

In [ ]:
def evaluate(score_fn,k=20,n_eval=2000):
    users=np.where(np.diff(test.indptr)>0)[0][:n_eval]
    rec,popm=[],[]
    for u in users:
        s=np.array(score_fn(u),dtype=float); s[train[u].indices]=-1e9
        top=np.argsort(-s)[:k]
        tset=set(test[u].indices)
        rec.append(len(set(top)&tset)/max(len(tset),1))
        popm.append(pop_pct[top].mean())
    return np.mean(rec), np.mean(popm)

print(f"{'Model':<26}{'Recall@20':>11}{'AvgPop%@20':>13}")
for name,fn in MODELS.items():
    r,p=evaluate(fn)
    print(f'{name:<26}{r:>11.4f}{p*100:>12.1f}%')
print('\n=> DIVRS: AvgPop% THAP hon Most-Popular = it hua theo do pho bien (da khu thien lech).')

## B. Đường cong: độ phổ biến TB của gợi ý theo Top-K (thấp = khử thiên lệch tốt)

In [ ]:
import matplotlib.pyplot as plt
Ks=[5,10,20,30,40,50]
plt.figure(figsize=(6,4))
for name,fn in MODELS.items():
    y=[evaluate(fn,k=k,n_eval=1000)[1]*100 for k in Ks]
    plt.plot(Ks,y,marker='o',label=name)
plt.xlabel('Top-K'); plt.ylabel('Do pho bien TB cua goi y (%)')
plt.title('Khu thien lech: DIVRS thap hon baseline'); plt.legend(); plt.grid(alpha=.3)
plt.savefig('debias_curve.png',dpi=120,bbox_inches='tight'); plt.show()
print('Da luu debias_curve.png')

## C. WEB DEMO (Gradio) — chọn user, xem gợi ý baseline vs DIVRS
Chạy xong in ra **link public** (xxx.gradio.live) — mở trình duyệt để demo.

In [ ]:
import gradio as gr
def reco(uid,k):
    uid=int(uid); k=int(k); res=[]
    for name,fn in MODELS.items():
        s=np.array(fn(uid),dtype=float); s[train[uid].indices]=-1e9
        top=np.argsort(-s)[:k]
        lines=[f'#{r+1:>2}  Item {it:<5}  do pho bien: {pop_pct[it]*100:>4.0f}%' for r,it in enumerate(top)]
        res.append('\n'.join(lines)+f'\n\n>>> Do pho bien TB: {pop_pct[top].mean()*100:.0f}%')
    return res[0],res[1]

gr.Interface(reco,
    [gr.Slider(0,n_user-1,step=1,value=0,label='User ID'),
     gr.Slider(5,20,step=1,value=10,label='Top-K')],
    [gr.Textbox(label='Most-Popular baseline (pho bien cao)',lines=12),
     gr.Textbox(label='DIVRS (da khu thien lech)',lines=12)],
    title='DIVRS Debiasing Demo — ML-10M',
    description='Keo chon user. DIVRS co do pho bien TB thap hon baseline = it hua theo dam dong.'
).launch(share=True)

## Ghi chú
- Baseline = **Most-Popular** (không train) — đại diện cực điểm thiên lệch đám đông.
- Muốn baseline **MF** học thật như paper: train riêng `--flagfile config/ml10m_mf.cfg` rồi nạp thêm.
- Item theo **ID** (data đã reindex). Muốn tên phim: tải `movies.dat` + map `item_reindex.json`.
- `AvgPop%` = proxy cho IOU. Thấp hơn = khử thiên lệch tốt hơn.